# Exp4 — PhoBERT-v2 vs XLM-RoBERTa Comparison

**Mục tiêu:** So sánh PhoBERT-v2 với XLM-R trên cùng điều kiện (Exp2 setup: Teencode normalization, no tabular branch, cùng train/val/test split seed=42).

| Exp | Backbone | Teencode | Tabular | Kết quả trước |
|---|---|---|---|---|
| Exp2 | **XLM-RoBERTa-base** | ✅ | ❌ | F1=0.6548 |
| Exp4 | **PhoBERT-base-v2** | ✅ | ❌ | ← chạy notebook này |

**Thời gian ước tính:** ~10 phút trên Colab T4.

## 0. Setup môi trường

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Chỉnh path này nếu repo nằm ở chỗ khác trên Drive ──
PROJECT_DIR = '/content/drive/MyDrive/deep-social-sentiment-analysis'

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Files:', [f for f in os.listdir('.') if not f.startswith('.')])

In [ ]:
%%capture
# Cài thêm nếu thiếu (thường đã có từ lần train trước)
!pip install -q transformers accelerate rtdl-revisiting-models einops lime sentencepiece

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Chạy Exp4 (PhoBERT)

Chỉ chạy Exp4 — không re-train Exp1/2/3 (đã có kết quả rồi).

In [ ]:
import sys
import logging
import time
from pathlib import Path

sys.path.insert(0, PROJECT_DIR)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from scripts.run_ablation import (
    load_raw, load_uit_vsmec, make_text_derived_features, run_experiment,
    CLASS_NAMES,
)
from src.preprocessing import TeencodeNormalizer, stratified_split

# ── Config ──────────────────────────────────────────────────────────────
SEED        = 42
EPOCHS      = 4
BATCH_SIZE  = 16
LR          = 2e-5
MAX_LENGTH  = 256   # PhoBERT tends to produce longer token sequences
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_ROOT = Path('models/ablation')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RAW_PATH    = Path('data/raw/crawled_emotions.xlsx')
VSMEC_PATH  = Path('data/raw/UIT-VSMEC.csv')

print(f'Device: {DEVICE}')
print(f'PhoBERT max_length: {MAX_LENGTH}')

In [ ]:
# ── Load + merge dữ liệu (giống hệt run_ablation.py main()) ─────────────
raw = load_raw(RAW_PATH)
vsmec = load_uit_vsmec(VSMEC_PATH)
if not vsmec.empty:
    raw = pd.concat([raw, vsmec], ignore_index=True)
    print(f'Merged UIT-VSMEC → total {len(raw)} rows')

norm = TeencodeNormalizer()
raw['norm_text'] = norm.transform(raw['raw_text'])
raw = raw[raw['norm_text'].str.len() > 0].reset_index(drop=True)

behavior = make_text_derived_features(raw['norm_text'])
raw = pd.concat([raw, behavior], axis=1)

train_df, val_df, test_df = stratified_split(
    raw, label_column='label',
    train_size=0.70, val_size=0.15, test_size=0.15,
    seed=SEED,
)
print(f'Splits: train={len(train_df)} | val={len(val_df)} | test={len(test_df)}')
print('Label dist (train):')
print(train_df['label'].value_counts().to_string())

In [ ]:
# ── Exp4: PhoBERT-v2 + Teencode, no tabular ─────────────────────────────
# Đây là bản so sánh fair với Exp2 (XLM-R + Teencode, no tabular).
# Chỉ thay backbone, mọi thứ khác giữ nguyên.

print('=' * 60)
print('Running Exp4: PhoBERT-v2 vs XLM-R')
print('Backbone: vinai/phobert-base-v2')
print('Conditions: Teencode ✅ | Tabular ❌ | seed=42')
print('=' * 60)

t_start = time.time()

result_exp4 = run_experiment(
    name='exp4_phobert',
    train_df=train_df, val_df=val_df, test_df=test_df,
    use_normalizer=False,
    use_tabular=False,
    text_col='norm_text',
    text_model_name='vinai/phobert-base-v2',
    max_length=MAX_LENGTH,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
)

elapsed = time.time() - t_start
print(f'\nExp4 done in {elapsed/60:.1f} min')
print(f"F1-Macro  : {result_exp4['f1_macro']:.4f}")
print(f"Accuracy  : {result_exp4['accuracy']:.4f}")
print(f"Checkpoint: {result_exp4['checkpoint']}")

## 2. So sánh với Exp1–3

In [ ]:
import pandas as pd

# Load Exp1–3 từ file đã lưu
old = pd.read_csv('reports/ablation_results.csv', index_col='experiment')

# Build Exp4 row
new_row = pd.DataFrame([{
    'experiment':      'exp4_phobert',
    'text_model':      'vinai/phobert-base-v2',
    'use_normalizer':  result_exp4['use_normalizer'],
    'use_tabular':     result_exp4['use_tabular'],
    'best_epoch':      result_exp4['best_epoch'],
    'train_seconds':   result_exp4['train_seconds'],
    'f1_macro':        result_exp4['f1_macro'],
    'precision_macro': result_exp4['precision_macro'],
    'recall_macro':    result_exp4['recall_macro'],
    'accuracy':        result_exp4['accuracy'],
    'f1_weighted':     result_exp4['f1_weighted'],
}]).set_index('experiment')

# Add text_model to old if missing
if 'text_model' not in old.columns:
    xlmr = 'xlm-roberta-base'
    old.insert(0, 'text_model', xlmr)

combined = pd.concat([old, new_row])

# Pretty display
display_cols = ['text_model', 'use_normalizer', 'use_tabular',
                'f1_macro', 'precision_macro', 'recall_macro', 'accuracy', 'best_epoch']
display_df = combined[display_cols].copy()
display_df['text_model'] = display_df['text_model'].str.replace('xlm-roberta-base', 'XLM-R').str.replace('vinai/phobert-base-v2', 'PhoBERT-v2')

for col in ['f1_macro', 'precision_macro', 'recall_macro', 'accuracy']:
    display_df[col] = display_df[col].map('{:.4f}'.format)

print('\n=== ABLATION RESULTS (test split) ===')
print(display_df.to_string())

# Highlight: Exp2 vs Exp4 (fair comparison)
exp2_f1 = float(old.loc['exp2_xlmr_teencode', 'f1_macro'])
exp4_f1 = result_exp4['f1_macro']
delta   = exp4_f1 - exp2_f1
winner  = 'PhoBERT' if delta > 0 else 'XLM-R'
print(f'\n── XLM-R (Exp2) F1-Macro : {exp2_f1:.4f}')
print(f'── PhoBERT (Exp4) F1-Macro: {exp4_f1:.4f}')
print(f'── Delta: {delta:+.4f}  →  {winner} thắng')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))

labels = [
    'Exp1\nXLM-R only',
    'Exp2\nXLM-R+Teencode',
    'Exp3\nFull Fusion',
    'Exp4\nPhoBERT+Teencode',
]
exp_keys = ['exp1_xlmr_only', 'exp2_xlmr_teencode', 'exp3_full_fusion', 'exp4_phobert']
f1_vals = [float(combined.loc[k, 'f1_macro']) for k in exp_keys]
acc_vals = [float(combined.loc[k, 'accuracy']) for k in exp_keys]

x = np.arange(len(labels))
w = 0.35
colors_f1  = ['#6366f1', '#6366f1', '#6366f1', '#f59e0b']
colors_acc = ['#a5b4fc', '#a5b4fc', '#a5b4fc', '#fcd34d']

bars1 = ax.bar(x - w/2, f1_vals,  w, color=colors_f1,  label='F1-Macro',  zorder=3)
bars2 = ax.bar(x + w/2, acc_vals, w, color=colors_acc, label='Accuracy', zorder=3)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylim(0.55, 0.80)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Ablation Study: F1-Macro & Accuracy by Experiment\n(Exp4 = PhoBERT-v2, same conditions as Exp2)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4, zorder=0)

# Mark Exp2 vs Exp4 comparison
ax.annotate('', xy=(x[3] - w/2, f1_vals[3] + 0.025),
            xytext=(x[1] - w/2, f1_vals[1] + 0.025),
            arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
mid = (x[1] + x[3]) / 2
ax.text(mid - w/2, max(f1_vals[1], f1_vals[3]) + 0.032,
        f'Δ={delta:+.4f}', ha='center', color='red', fontsize=9, fontweight='bold')

plt.tight_layout()
out_path = 'reports/figures/ablation_with_phobert.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

## 3. Lưu kết quả

In [ ]:
# Lưu bảng đầy đủ (Exp1–4) vào reports/
save_cols = ['text_model', 'use_normalizer', 'use_tabular', 'best_epoch',
             'train_seconds', 'f1_macro', 'precision_macro',
             'recall_macro', 'accuracy', 'f1_weighted']

# Only keep columns that exist
save_cols = [c for c in save_cols if c in combined.columns]

combined[save_cols].to_csv('reports/ablation_results_with_phobert.csv')
combined[save_cols].to_markdown('reports/ablation_results_with_phobert.md')

print('Saved:')
print('  reports/ablation_results_with_phobert.csv')
print('  reports/ablation_results_with_phobert.md')
print('  reports/figures/ablation_with_phobert.png')
print()
print('Done! Copy kết quả Exp4 vào PROJECT_ANALYSIS.md và notebook 02.')

## 4. Kết luận cho bảo vệ

Dùng kết quả này để trả lời câu hỏi giảng viên:

> **"Tại sao dùng XLM-R mà không phải PhoBERT?"**

Nếu XLM-R thắng:
> *XLM-R được pre-train trên 100 ngôn ngữ bao gồm tiếng Việt và xử lý code-switching tốt hơn. PhoBERT cần VnCoreNLP word segmentation — không có segmenter đúng thì performance giảm. Ablation table cho thấy XLM-R + Teencode đạt F1-Macro=0.6548 vs PhoBERT+Teencode=X.XXXX.*

Nếu PhoBERT thắng:
> *PhoBERT được fine-tune chuyên biệt cho tiếng Việt đạt F1-Macro cao hơn. Tuy nhiên XLM-R vẫn được chọn cho production vì không phụ thuộc vào VnCoreNLP segmenter — dễ deploy hơn.*